## 19. Fine-Tuning parcial de wav2vec2 para DDK-PD

En vez de usar embeddings congelados, descongelamos las últimas capas del transformer de wav2vec2 para que aprenda representaciones específicas para diadococinesia en Parkinson.

In [1]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2Model, Wav2Vec2Processor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score, roc_curve
from sklearn.preprocessing import StandardScaler

TARGET_SR = 16_000
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
RANDOM_STATE = 42
N_FOLDS = 5

print(f"Device: {DEVICE}")

/Users/napster/Documents/upm/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps


In [2]:
from src.preprocessing import load_waveforms, preprocess_waveform, load_metadata
from src.spectral import extract_spectral_features

df_metadata = load_metadata(base_path="data")
waveforms_raw, df_metadata = load_waveforms(df_metadata, target_sr=TARGET_SR)
waveforms_processed = [preprocess_waveform(wf) for wf in waveforms_raw]
labels_all = df_metadata["label"].to_numpy()

# Spectral features (rama espectral sigue congelada)
MEL_CONFIG = {"sample_rate": TARGET_SR, "n_fft": 1024, "hop_length": 512, "n_mels": 128}
spectral_features = extract_spectral_features(waveforms_processed, device=DEVICE, **MEL_CONFIG)

scaler_s = StandardScaler()
spectral_all = scaler_s.fit_transform(spectral_features)

print(f"Sujetos: {len(waveforms_processed)}")
print(f"Spectral features: {spectral_all.shape}")

2026-04-19 23:24:23,455 [INFO] src.config: Generando mel-espectrogramas...
2026-04-19 23:24:23,520 [INFO] src.config:   100 espectrogramas generados
2026-04-19 23:24:23,520 [INFO] src.config:   Shape ejemplo: (128, 107)
2026-04-19 23:24:23,520 [INFO] src.config: Extrayendo features con ResNet18 (ImageNet)...
2026-04-19 23:24:23,984 [INFO] src.config:   25/100
2026-04-19 23:24:24,052 [INFO] src.config:   50/100
2026-04-19 23:24:24,117 [INFO] src.config:   75/100
2026-04-19 23:24:24,181 [INFO] src.config:   100/100
2026-04-19 23:24:24,182 [INFO] src.config:   Features espectrales: (100, 512)


Sujetos: 100
Spectral features: (100, 512)


### Dataset para fine-tuning

A diferencia del pipeline anterior, el dataset ahora devuelve waveforms crudos (que wav2vec2 procesa en cada forward pass) + spectral features pre-extraídos.

In [3]:
class FineTuneDataset(Dataset):
    """Dataset que devuelve waveforms + spectral features + labels."""
    def __init__(self, waveforms, spectral, labels, indices):
        self.waveforms = [waveforms[i] for i in indices]
        self.spectral = torch.tensor(spectral[indices], dtype=torch.float32)
        self.labels = torch.tensor(labels[indices], dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        wf = torch.tensor(self.waveforms[idx], dtype=torch.float32)
        return wf, self.spectral[idx], self.labels[idx]


def collate_fn(batch):
    """Pad waveforms al máximo largo del batch."""
    waveforms, spectral, labels = zip(*batch)
    max_len = max(w.shape[0] for w in waveforms)
    wf_padded = torch.zeros(len(waveforms), max_len)
    attention_mask = torch.zeros(len(waveforms), max_len, dtype=torch.long)
    for i, w in enumerate(waveforms):
        wf_padded[i, :w.shape[0]] = w
        attention_mask[i, :w.shape[0]] = 1
    return wf_padded, attention_mask, torch.stack(spectral), torch.stack(labels)

print("Dataset y collate_fn definidos.")

Dataset y collate_fn definidos.


### Modelo end-to-end: wav2vec2 (parcialmente descongelado) + DBFNet

In [4]:
class FineTuneDBFNet(nn.Module):
    """
    wav2vec2 con últimas N capas descongeladas + DBFNet classifier.
    La rama temporal usa wav2vec2 end-to-end.
    La rama espectral usa features pre-extraídos (congelados).
    """
    def __init__(self, model_name="facebook/wav2vec2-base",
                 n_unfreeze_layers=2, dim_proj=128, dim_hidden=64,
                 dropout=0.3, dim_spectral=512):
        super().__init__()

        # Cargar wav2vec2
        self.wav2vec2 = Wav2Vec2Model.from_pretrained(model_name)

        # Congelar todo
        for param in self.wav2vec2.parameters():
            param.requires_grad = False

        # Descongelar últimas N capas del encoder transformer
        total_layers = len(self.wav2vec2.encoder.layers)
        for i in range(total_layers - n_unfreeze_layers, total_layers):
            for param in self.wav2vec2.encoder.layers[i].parameters():
                param.requires_grad = True

        dim_temporal = self.wav2vec2.config.hidden_size  # 768

        # Proyecciones (igual que DBFNet original)
        self.proj_temporal = nn.Sequential(
            nn.Linear(dim_temporal, dim_proj),
            nn.LayerNorm(dim_proj),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.proj_spectral = nn.Sequential(
            nn.Linear(dim_spectral, dim_proj),
            nn.LayerNorm(dim_proj),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.gate = nn.Sequential(
            nn.Linear(dim_proj * 2, dim_proj),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_proj, 1),
            nn.Sigmoid(),
        )
        self.classifier = nn.Sequential(
            nn.Linear(dim_proj, dim_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_hidden, 1),
        )

    def forward(self, waveform, attention_mask, z_spectral):
        # wav2vec2 forward: waveform → embeddings temporales
        w2v_out = self.wav2vec2(waveform, attention_mask=attention_mask)
        # Mean pooling sobre la secuencia temporal
        hidden = w2v_out.last_hidden_state  # (batch, seq_len, 768)
        # Usar attention_mask para mean pooling correcto
        mask = attention_mask.unsqueeze(-1).float()
        # Calcular cuántos frames genera wav2vec2
        seq_len = hidden.shape[1]
        mask_resampled = torch.nn.functional.interpolate(
            mask.transpose(1, 2), size=seq_len, mode='nearest'
        ).transpose(1, 2)
        z_temporal = (hidden * mask_resampled).sum(dim=1) / mask_resampled.sum(dim=1).clamp(min=1)

        # Fusión (igual que DBFNet)
        h_t = self.proj_temporal(z_temporal)
        h_s = self.proj_spectral(z_spectral)
        concat = torch.cat([h_t, h_s], dim=1)
        alpha = self.gate(concat)
        h_fused = alpha * h_t + (1 - alpha) * h_s
        logits = self.classifier(h_fused).squeeze(-1)
        return logits, alpha


# Verificar parámetros
_test_model = FineTuneDBFNet(n_unfreeze_layers=2)
total = sum(p.numel() for p in _test_model.parameters())
trainable = sum(p.numel() for p in _test_model.parameters() if p.requires_grad)
print(f"Parámetros totales: {total:,}")
print(f"Parámetros entrenables: {trainable:,} ({100*trainable/total:.1f}%)")
del _test_model

2026-04-19 23:24:24,562 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-19 23:24:24,629 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/wav2vec2-base/0b5b8e868dd84f03fd87d01f9c4ff0f080fecfe8/config.json "HTTP/1.1 200 OK"
2026-04-19 23:24:24,735 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-base/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
2026-04-19 23:24:24,838 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-base/resolve/main/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-04-19 23:24:24,944 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-base/resolve/main/pytorch_model.bin "HTTP/1.1 302 Found"
2026-04-19 23:24:25,049 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-base/resolve/main/model.safetensors "HTTP/1.1 404 Not Fou

Parámetros totales: 94,577,666
Parámetros entrenables: 14,381,698 (15.2%)


2026-04-19 23:24:25,400 [INFO] httpx: HTTP Request: GET https://huggingface.co/api/models/facebook/wav2vec2-base/discussions?p=0 "HTTP/1.1 200 OK"
2026-04-19 23:24:25,513 [INFO] httpx: HTTP Request: GET https://huggingface.co/api/models/facebook/wav2vec2-base/commits/refs%2Fpr%2F11 "HTTP/1.1 200 OK"
2026-04-19 23:24:25,615 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-base/resolve/refs%2Fpr%2F11/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-04-19 23:24:25,719 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-base/resolve/refs%2Fpr%2F11/model.safetensors "HTTP/1.1 302 Found"


### Función de entrenamiento para fine-tuning

In [5]:
def train_fold_finetune(train_ds, val_ds, device,
                        lr_classifier=7e-4, lr_wav2vec=1e-5,
                        weight_decay=3e-4, dropout=0.3,
                        dim_proj=128, dim_hidden=64,
                        n_unfreeze_layers=2,
                        epochs=60, patience=10, batch_size=4):
    """
    Entrena un fold con fine-tuning parcial de wav2vec2.
    Usa learning rates diferentes para wav2vec2 y clasificador.
    """
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=len(val_ds), shuffle=False, collate_fn=collate_fn)

    model = FineTuneDBFNet(
        n_unfreeze_layers=n_unfreeze_layers,
        dim_proj=dim_proj, dim_hidden=dim_hidden, dropout=dropout
    ).to(device)

    # Learning rates diferenciados
    wav2vec_params = [p for n, p in model.named_parameters()
                      if 'wav2vec2' in n and p.requires_grad]
    classifier_params = [p for n, p in model.named_parameters()
                         if 'wav2vec2' not in n and p.requires_grad]

    optimizer = torch.optim.AdamW([
        {'params': wav2vec_params, 'lr': lr_wav2vec},
        {'params': classifier_params, 'lr': lr_classifier},
    ], weight_decay=weight_decay)

    criterion = nn.BCEWithLogitsLoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5
    )

    best_val_loss = float('inf')
    epochs_no_improve = 0
    best_state = None
    history = {'train_loss': [], 'val_loss': [], 'lr': []}

    for epoch in range(epochs):
        model.train()
        train_losses = []
        for wf, mask, z_s, y in train_loader:
            wf, mask = wf.to(device), mask.to(device)
            z_s, y = z_s.to(device), y.to(device)

            logits, _ = model(wf, mask, z_s)
            loss = criterion(logits, y)

            optimizer.zero_grad()
            loss.backward()
            # Gradient clipping para estabilidad
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            wf_v, mask_v, z_s_v, y_v = next(iter(val_loader))
            wf_v, mask_v = wf_v.to(device), mask_v.to(device)
            z_s_v, y_v = z_s_v.to(device), y_v.to(device)
            logits_v, alphas_v = model(wf_v, mask_v, z_s_v)
            val_loss = criterion(logits_v, y_v).item()

        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[1]['lr']  # classifier lr
        history['train_loss'].append(np.mean(train_losses))
        history['val_loss'].append(val_loss)
        history['lr'].append(current_lr)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        wf_v, mask_v, z_s_v, y_v = next(iter(val_loader))
        wf_v, mask_v = wf_v.to(device), mask_v.to(device)
        z_s_v, y_v = z_s_v.to(device), y_v.to(device)
        logits_v, alphas_v = model(wf_v, mask_v, z_s_v)
        y_prob = torch.sigmoid(logits_v).cpu().numpy()
        y_true = y_v.cpu().numpy()
        alphas = alphas_v.cpu().numpy()

    return y_true, y_prob, alphas, history

print("Función de entrenamiento definida.")

Función de entrenamiento definida.


### Experimento: 3 configuraciones de fine-tuning

In [6]:
CONFIGS = [
    {"name": "1 capa, lr=1e-5", "n_unfreeze_layers": 1, "lr_wav2vec": 1e-5},
    {"name": "2 capas, lr=5e-6", "n_unfreeze_layers": 2, "lr_wav2vec": 5e-6},
    {"name": "2 capas, lr=1e-5", "n_unfreeze_layers": 2, "lr_wav2vec": 1e-5},
]

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
all_config_results = {}

for config in CONFIGS:
    print(f"\n{'='*60}")
    print(f"Config: {config['name']}")
    print(f"{'='*60}")

    fold_aucs, fold_accs, fold_sens, fold_spec = [], [], [], []
    fold_histories = []

    for fold_idx, (tr_idx, val_idx) in enumerate(skf.split(np.arange(len(labels_all)), labels_all)):
        print(f"  Fold {fold_idx+1}/{N_FOLDS}...", end=" ")

        torch.manual_seed(RANDOM_STATE + fold_idx)
        np.random.seed(RANDOM_STATE + fold_idx)

        train_ds = FineTuneDataset(waveforms_processed, spectral_all, labels_all, tr_idx)
        val_ds = FineTuneDataset(waveforms_processed, spectral_all, labels_all, val_idx)

        y_true, y_prob, alphas, history = train_fold_finetune(
            train_ds, val_ds, DEVICE,
            lr_classifier=7e-4,
            lr_wav2vec=config["lr_wav2vec"],
            weight_decay=3e-4,
            dropout=0.3,
            dim_proj=128, dim_hidden=64,
            n_unfreeze_layers=config["n_unfreeze_layers"],
            epochs=60, patience=10, batch_size=4,
        )

        y_pred = (y_prob >= 0.5).astype(int)
        auc = roc_auc_score(y_true, y_prob)
        acc = accuracy_score(y_true, y_pred)
        sens = recall_score(y_true, y_pred, pos_label=1)
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        spec = 1 - fpr[np.argmax(tpr - fpr)]

        fold_aucs.append(auc)
        fold_accs.append(acc)
        fold_sens.append(sens)
        fold_spec.append(spec)
        fold_histories.append(history)

        print(f"AUC={auc:.3f} | Acc={acc:.3f} | Sens={sens:.3f} | Spec={spec:.3f}")

    all_config_results[config['name']] = {
        'aucs': fold_aucs, 'accs': fold_accs,
        'sens': fold_sens, 'spec': fold_spec,
        'histories': fold_histories,
    }

    print(f"\n  Media: AUC={np.mean(fold_aucs):.3f}±{np.std(fold_aucs):.3f} | "
          f"Acc={np.mean(fold_accs):.3f}±{np.std(fold_accs):.3f} | "
          f"Sens={np.mean(fold_sens):.3f}±{np.std(fold_sens):.3f} | "
          f"Spec={np.mean(fold_spec):.3f}±{np.std(fold_spec):.3f}")


Config: 1 capa, lr=1e-5
  Fold 1/5... 

2026-04-19 23:24:31,170 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-19 23:24:31,238 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/wav2vec2-base/0b5b8e868dd84f03fd87d01f9c4ff0f080fecfe8/config.json "HTTP/1.1 200 OK"
2026-04-19 23:24:31,347 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-base/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
2026-04-19 23:24:31,468 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-base/resolve/main/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-04-19 23:24:31,621 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-base/resolve/main/pytorch_model.bin "HTTP/1.1 302 Found"
2026-04-19 23:24:31,726 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-base/resolve/main/model.safetensors "HTTP/1.1 404 Not Fou

AUC=0.810 | Acc=0.800 | Sens=0.600 | Spec=1.000
  Fold 2/5... 

2026-04-19 23:27:18,587 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-19 23:27:18,659 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/wav2vec2-base/0b5b8e868dd84f03fd87d01f9c4ff0f080fecfe8/config.json "HTTP/1.1 200 OK"
2026-04-19 23:27:18,785 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-base/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
2026-04-19 23:27:18,901 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-base/resolve/main/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-04-19 23:27:19,009 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-base/resolve/main/pytorch_model.bin "HTTP/1.1 302 Found"
2026-04-19 23:27:19,123 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-base/resolve/main/model.safetensors "HTTP/1.1 404 Not Fou

AUC=0.820 | Acc=0.700 | Sens=0.700 | Spec=0.900
  Fold 3/5... 

...

AUC=0.670 | Acc=0.550 | Sens=0.500 | Spec=0.500
  Fold 4/5... 

2026-04-19 23:34:56,864 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-19 23:34:56,937 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/wav2vec2-base/0b5b8e868dd84f03fd87d01f9c4ff0f080fecfe8/config.json "HTTP/1.1 200 OK"
2026-04-19 23:34:57,089 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-base/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
2026-04-19 23:34:57,203 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-base/resolve/main/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-04-19 23:34:57,324 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-base/resolve/main/pytorch_model.bin "HTTP/1.1 302 Found"
2026-04-19 23:34:57,440 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-base/resolve/main/model.safetensors "HTTP/1.1 404 Not Fou

AUC=0.740 | Acc=0.800 | Sens=0.900 | Spec=0.700
  Fold 5/5... 

2026-04-19 23:42:59,070 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-19 23:42:59,142 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/wav2vec2-base/0b5b8e868dd84f03fd87d01f9c4ff0f080fecfe8/config.json "HTTP/1.1 200 OK"
2026-04-19 23:42:59,284 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-base/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
2026-04-19 23:42:59,391 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-base/resolve/main/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-04-19 23:42:59,497 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-base/resolve/main/pytorch_model.bin "HTTP/1.1 302 Found"
2026-04-19 23:42:59,604 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-base/resolve/main/model.safetensors "HTTP/1.1 404 Not Fou

KeyboardInterrupt: 

### Resultados: Fine-tuning vs Baseline congelado

In [ ]:
import matplotlib.pyplot as plt

# Baseline congelado (del notebook anterior)
BASELINE = {"AUC": 0.902, "Acc": 0.890, "Sens": 0.840, "Spec": 0.940}

print("="*75)
print("Comparación: Fine-tuning vs Baseline congelado (5-fold CV)")
print("="*75)
print(f"  {'Config':<25} {'AUC':>15} {'Acc':>15} {'Sens':>15} {'Spec':>15}")
print(f"  {'─'*70}")
print(f"  {'Baseline (congelado)':<25} {BASELINE['AUC']:>15.3f} {BASELINE['Acc']:>15.3f} {BASELINE['Sens']:>15.3f} {BASELINE['Spec']:>15.3f}")

for name, res in all_config_results.items():
    print(f"  {name:<25} {np.mean(res['aucs']):>7.3f}±{np.std(res['aucs']):.3f} "
          f"{np.mean(res['accs']):>7.3f}±{np.std(res['accs']):.3f} "
          f"{np.mean(res['sens']):>7.3f}±{np.std(res['sens']):.3f} "
          f"{np.mean(res['spec']):>7.3f}±{np.std(res['spec']):.3f}")

# Gráfico de barras
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
metrics = ['AUC', 'Acc', 'Sens', 'Spec']
keys = ['aucs', 'accs', 'sens', 'spec']
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']

config_names = ['Baseline\n(congelado)'] + list(all_config_results.keys())

for ax, metric, key, color in zip(axes, metrics, keys, colors):
    vals = [BASELINE[metric]]
    errs = [0]
    for name, res in all_config_results.items():
        vals.append(np.mean(res[key]))
        errs.append(np.std(res[key]))

    x = np.arange(len(vals))
    bars = ax.bar(x, vals, yerr=errs, capsize=5, color=color, alpha=0.7)
    ax.set_xticks(x)
    ax.set_xticklabels(config_names, fontsize=8, rotation=15)
    ax.set_title(metric)
    ax.set_ylim(0.5, 1.05)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', fontsize=8)

plt.suptitle('Fine-tuning wav2vec2 vs Baseline congelado', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### Curvas de pérdida por fold (mejor configuración)

In [ ]:
# Seleccionar mejor configuración
best_config = max(all_config_results, key=lambda k: np.mean(all_config_results[k]['aucs']))
print(f"Mejor configuración: {best_config}")

histories = all_config_results[best_config]['histories']
fold_aucs = all_config_results[best_config]['aucs']

fig, axes = plt.subplots(1, N_FOLDS, figsize=(20, 4), sharey=True)

for i, (ax, hist) in enumerate(zip(axes, histories)):
    epochs = range(1, len(hist['train_loss']) + 1)
    ax.plot(epochs, hist['train_loss'], label='Train', linewidth=1.5, color='#2196F3')
    ax.plot(epochs, hist['val_loss'], label='Val', linewidth=1.5, color='#F44336')
    ax.set_title(f'Fold {i+1} (AUC={fold_aucs[i]:.3f})', fontsize=10)
    ax.set_xlabel('Epoch')
    if i == 0:
        ax.set_ylabel('Loss')
    ax.legend(fontsize=8)

plt.suptitle(f'Curvas de pérdida — Fine-tuning ({best_config})', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### Conclusiones

*(Completar tras ejecutar)*